# Chapter 11 — Hidden Notebook State

**Book alignment:** Debugging AI From First Principles, Chapter 11

**Question this notebook isolates:** The warm kernel answers `threshold == 0.9`; no visible
cell defines `threshold`. Does the fresh-vs-dirty kernel diff separate **H1** (deleted-cell
residue — absent in a fresh run, `NameError`) from **H2** (overwritten-but-not-rerun — present
in a fresh run, but a *different* value)?

A `.ipynb` is modelled as a list of cell dicts plus a dirty namespace, exactly as the kernel
holds them.

In [ ]:
# the notebook as stored: visible source + the execution_count the kernel stamped.
# plus DIRTY: the warm kernel's namespace, carrying residue from executions since edited away.
VISIBLE_CELLS = [
    {"src": "import pandas as pd",                       "exec": 1},
    {"src": "df = pd.read_csv('sales.csv')",             "exec": 2},
    {"src": "df = df.dropna()",                          "exec": 3},
    {"src": "threshold = load_config()['cutoff']",       "exec": 9},   # EDITED text, never re-run
    {"src": "flagged = df[df.score > threshold]",        "exec": 10},
    {"src": "print(threshold)",                          "exec": 11},
]
DIRTY_NAMESPACE = {"pd": "<module>", "df": "<DataFrame>", "threshold": 0.9, "flagged": "<DataFrame>"}

# history the kernel remembers but the document forgot:
KERNEL_HISTORY = [(8, "threshold = 0.9"), (9, "threshold = load_config()['cutoff']  # edited, not re-run")]
CONFIG = {"cutoff": 0.7}          # what load_config() actually returns today

## 1. Dump the dirty kernel; search the visible text — the name is nowhere

In [ ]:
suspect = "threshold"
visible_defs = [c for c in VISIBLE_CELLS if c["src"].startswith(f"{suspect} =") or f"{suspect}=" in c["src"].replace(" ", "")]
print(f"dirty kernel: {suspect} = {DIRTY_NAMESPACE[suspect]!r} ({type(DIRTY_NAMESPACE[suspect]).__name__})")
print(f"visible cells defining {suspect}: {[c['exec'] for c in visible_defs]}")
# there IS a visible line (exec 9) but it was never run after the edit -> its value is a fossil
assert suspect in DIRTY_NAMESPACE
print("a value in the kitchen; the recipe line was edited and never re-cooked")

## 2. The discriminating experiment: restart, run the visible prefix once

In [ ]:
def fresh_run(cells, config):
    ns = {}
    for c in cells:
        src = c["src"]
        if src.startswith("threshold ="):
            ns["threshold"] = config["cutoff"]          # the CURRENT text, executed honestly
        elif src.startswith("flagged ="):
            if "threshold" not in ns:
                return ("NameError", "threshold")       # H1 outcome: no live definition exists
        # other lines: modelled as no-ops for the suspect
    return ("ok", ns.get("threshold", None))

# H2 scenario (as stored above): exec-9 line is present, so a fresh kernel DEFINES threshold ...
status, value = fresh_run(VISIBLE_CELLS, CONFIG)
print(f"fresh kernel: status={status}  threshold={value}")
assert status == "ok" and value == 0.7
assert value != DIRTY_NAMESPACE["threshold"]           # 0.7 (fresh, correct) != 0.9 (dirty fossil)
print("present in both kernels, DIFFERENT value -> H2: overwritten-but-not-rerun")

# H1 scenario: delete the definition line entirely -> fresh kernel raises
h1_cells = [c for c in VISIBLE_CELLS if not c["src"].startswith("threshold =")]
assert fresh_run(h1_cells, CONFIG) == ("NameError", "threshold")
print("delete the line -> fresh kernel NameErrors -> H1: deleted-cell residue")

## 3. Name the provenance line — a pointer, not a story

In [ ]:
orphan = [h for h in KERNEL_HISTORY if h[0] not in {c["exec"] for c in VISIBLE_CELLS}]
print("orphaned execution (in history, not in any visible cell):", orphan)
assert orphan == [(8, "threshold = 0.9")]
print("the 0.9 came from exec [8], deleted in cleanup. fix = restore the value in VISIBLE text,")
print("then verify with a fresh Restart & Run All - never `del threshold` on the warm kernel")

## What we earned

The kernel knew something no visible cell taught it. Dumping the dirty namespace and
searching the text located the anomaly; a fresh-kernel run of the visible prefix was the
experiment that classified it — **absent** (`NameError`) convicts deleted-cell residue,
**present-but-different** convicts an overwritten line never re-executed. The fix lands in
visible text and is gated by a green Restart & Run All, not a warm-kernel patch.

**Notebook 12 / Chapter 12** takes the green Run All and asks the next question: does it
*reproduce* — same numbers, another machine, another run?